Notebook to test on real data

In [3]:
import pandas as pd
import numpy as np

from datkit.cleaning import clean_dataframe

Import table

In [4]:
# don't use dtype=str as that creates legacy objects that pythons stores strings in
# pyarrow is a better option for string storage and manipulation (columnar storage)
df = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz", sep="\t", dtype_backend="pyarrow", nrows=100_000
)

clean table

In [5]:
df.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
43660,tt0044455,movie,Buffalo Bill in Tomahawk Territory,Buffalo Bill in Tomahawk Territory,0,1952,\N,66,"Drama,Western"
87278,tt0089267,movie,Heilende Schläge,Heilende Schläge,0,1985,\N,\N,\N
14317,tt0014555,movie,Toilers of the Sea,Toilers of the Sea,0,1923,\N,60,Drama
81932,tt0083769,movie,Sha ren ai qing jie,Sha ren ai qing jie,0,1982,\N,90,"Action,Romance"
95321,tt0097513,movie,Hisaab Khoon Ka,Hisaab Khoon Ka,0,1989,\N,\N,"Drama,Mystery"


In [6]:
df_clean, cleaning_report = clean_dataframe(df, False)

In [7]:
df_clean.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
43660,tt0044455,movie,Buffalo Bill in Tomahawk Territory,Buffalo Bill in Tomahawk Territory,0,1952,<NA>,66,"Drama,Western"
87278,tt0089267,movie,Heilende Schläge,Heilende Schläge,0,1985,<NA>,<NA>,<NA>
14317,tt0014555,movie,Toilers of the Sea,Toilers of the Sea,0,1923,<NA>,60,Drama
81932,tt0083769,movie,Sha ren ai qing jie,Sha ren ai qing jie,0,1982,<NA>,90,"Action,Romance"
95321,tt0097513,movie,Hisaab Khoon Ka,Hisaab Khoon Ka,0,1989,<NA>,<NA>,"Drama,Mystery"


In [8]:
cleaning_report

,column,dtype_before,dtype_after,converted,bytes_before,bytes_after,bytes_saved,mb_before,mb_after,mb_saved,nulls_before,nulls_after,nulls_created
0,tconst,string[pyarrow],string[pyarrow],False,1300000,1312500,-12500,1.24,1.25,-0.01,0,0,0
1,titleType,string[pyarrow],string[pyarrow],False,936793,949293,-12500,0.89,0.91,-0.01,0,0,0
2,primaryTitle,string[pyarrow],string[pyarrow],False,2139127,2151627,-12500,2.04,2.05,-0.01,0,0,0
3,originalTitle,string[pyarrow],string[pyarrow],False,2155967,2168467,-12500,2.06,2.07,-0.01,0,0,0
4,isAdult,int64[pyarrow],int64[pyarrow],False,800000,800000,0,0.76,0.76,0.00,0,0,0
5,startYear,string[pyarrow],int64[pyarrow],True,799974,812500,-12526,0.76,0.77,-0.01,0,13,13
6,endYear,string[pyarrow],int64[pyarrow],True,608384,812500,-204116,0.58,0.77,-0.19,0,95808,95808
7,runtimeMinutes,string[pyarrow],int64[pyarrow],True,613511,812500,-198989,0.59,0.77,-0.19,0,12376,12376
8,genres,string[pyarrow],string[pyarrow],False,1607178,1608584,-1406,1.53,1.53,-0.00,0,5547,5547


In [9]:
def sample_records(df: pd.DataFrame, n=10000, seed=1) -> pd.DataFrame:
    """Return a sample of records from the DataFrame."""
    if len(df) > n:
        return df.sample(n=n, random_state=seed)
    else:
        return df

In [10]:
df_clean_sample = sample_records(df_clean)

In [11]:
df_sample_nonunique = df_clean_sample.nunique()

In [12]:
df_sample_nonunique.iloc[2]

np.int64(9835)

In [13]:
df_sample_nonunique["isAdult"]

np.int64(2)

In [14]:
cleaning_report[cleaning_report["column"] == "isAdult"]["dtype_after"].values[0]

'int64[pyarrow]'

for col in df_clean_sample.columns:
    if df_sample_nonunique[col] > 100:
        if "string" in cleaning_report[cleaning_report["column"] == col]["dtype_after"].values[0]:
            pass
        else:
            df_clean_sample[col] = pd.cut(df_clean_sample[col], bins=5)
    elif df_sample_nonunique[col] > 10:
    else:

def binning(s:pd.Series) -> pd.Series:
    

In [16]:
df_clean_sample["runtimeMinutes_bins"] = pd.cut(df_clean_sample["runtimeMinutes"], bins=10)

In [17]:
df_clean_sample.groupby("runtimeMinutes_bins", observed=False)["runtimeMinutes"].agg(["count", "min", "max", "mean", "median", "std"])

,count,min,max,mean,median,std
runtimeMinutes_bins,,,,,,
"(-0.427, 143.7]",8549,1,143,74.047023,83.0,30.93211
"(143.7, 286.4]",178,144,282,187.747191,177.5,35.500905
"(286.4, 429.1]",33,288,420,335.606061,320.0,41.296897
"(429.1, 571.8]",3,476,540,512.0,520.0,32.741411
"(571.8, 714.5]",1,624,624,624.0,624.0,<NA>
"(714.5, 857.2]",1,763,763,763.0,763.0,<NA>
"(857.2, 999.9]",0,<NA>,<NA>,<NA>,<NA>,<NA>
"(999.9, 1142.6]",0,<NA>,<NA>,<NA>,<NA>,<NA>
"(1142.6, 1285.3]",0,<NA>,<NA>,<NA>,<NA>,<NA>


In [40]:
series_nona = df_clean_sample["runtimeMinutes"].dropna()
low, high = np.percentile(series_nona, [1,99]) # does not accept NA values
print(low, high)

6.0 210.0


In [26]:
arrasy_sample = np.array([1,2,2,3,4,4,4,5,5,6,6,7,7,8])
np.unique(arrasy_sample)

array([1, 2, 3, 4, 5, 6, 7, 8])

In [22]:
binresult = bin_column(df_clean_sample["titleType"], tail_min_span=0.3)
binresult

TypeError: bin_categorical() got an unexpected keyword argument 'tail_min_span'

In [53]:
binresult

,region,left,right,count,pct
0,low_tail,1.0,3.0,32,0.37
1,core,3.0,29.2,1035,11.81
2,core,29.2,55.4,883,10.07
3,core,55.4,81.6,2156,24.60
4,core,81.6,107.8,3731,42.56
5,core,107.8,134.0,639,7.29
6,core,134.0,160.2,124,1.41
7,core,160.2,186.4,55,0.63
8,core,186.4,212.6,26,0.30
9,core,212.6,238.8,22,0.25
